# Exemplo com a biblioteca Transformers e um modelo em Português

Neste exemplo, utilizaremos o modelo BERTimbau (BERT base em português) para prever palavras ocultas em uma frase.

In [1]:
!pip install -q transformers torch

In [ ]:
from transformers import pipeline

# Inicializa o pipeline de 'fill-mask' com o modelo BERTimbau
model_name = 'neuralmind/bert-base-portuguese-cased'
nlp = pipeline('fill-mask', model=model_name)

# Exemplo de frase com uma lacuna [MASK]
texto = "O Brasil é um país [MASK]."

# Executa a predição
resultados = nlp(texto)

# Exibe os resultados
for r in resultados:
    print(f"Sugestão: {r['sequence']} | Score: {r['score']:.4f}")

## Exemplo de Fine-Tuning em Português

Fine-tuning é o processo de ajustar um modelo pré-treinado em um conjunto de dados específico para uma tarefa nova.

Vamos demonstrar como preparar um modelo para classificar frases como 'Positiva' ou 'Negativa'.

In [3]:
!pip install -q datasets

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer, DataCollatorWithPadding
from datasets import Dataset
import torch

# Dados de exemplo (Sentimento: 0=Negativo, 1=Positivo)
data = {
    "text": [
        "Eu adorei este produto, ele é maravilhoso!",
        "Isso é horrível, nunca mais compro.",
        "O atendimento foi excelente e rápido.",
        "Estou muito insatisfeito com a qualidade.",
        "Valeu cada centavo, recomendo muito.",
        "Péssima experiência, odiei."
    ],
    "label": [1, 0, 1, 0, 1, 0]
}

dataset = Dataset.from_dict(data)

# Carregar Tokenizer e Modelo
model_checkpoint = "neuralmind/bert-base-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=2)

# Tokenização dos dados
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

tokenized_dataset = dataset.map(tokenize_function, batched=True)
tokenized_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

# Configurações de Treinamento
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    logging_steps=1,
    eval_strategy="no",
    report_to="none"
)

# Inicializar o Trainer com Data Collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

# Iniciar o treinamento
print("Iniciando fine-tuning corrigido...")
trainer.train()
print("Treinamento concluído!")

In [ ]:
frase_teste = "O produto chegou antes do prazo e funciona perfeitamente!"
inputs = tokenizer(frase_teste, return_tensors="pt", truncation=True, padding=True, max_length=128)

with torch.no_grad():
    logits = model(**inputs).logits

predicao = torch.argmax(logits, dim=1).item()
sentimento = "Positivo" if predicao == 1 else "Negativo"

print(f"Frase: {frase_teste}")
print(f"Sentimento previsto: {sentimento}")

Frase: O produto chegou antes do prazo e funciona perfeitamente!
Sentimento previsto: Positivo
